In [1]:
import composable.records as rec
from composable.strict import filter

In [2]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import plotnine as p9
import numpy as np

# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder, LabelBinarizer, StandardScaler

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Multiclass wrappers
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Metrics

# metric(y_test, y_predict)
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, RocCurveDisplay



In [4]:
(olive_oil :=
 pl.read_csv("data/OliveOils.csv")
).head(2)

Area.name,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
str,i64,i64,i64,i64,i64,i64,i64
"""North-Apulia""",1075,75,226,7823,672,36,60
"""North-Apulia""",1088,73,224,7709,781,31,61


In [5]:
(X_oil :=
 olive_oil
 .drop('Area.name')
 .to_pandas()
).head(2)

,palmitic,palmitoleic,strearic,oleic,linoleic,eicosanoic,linolenic
0,1075,75,226,7823,672,36,60
1,1088,73,224,7709,781,31,61


In [6]:
(y_oil :=
 olive_oil
 .select('Area.name')
 .to_numpy()
 .ravel()
)[:3]

array(['North-Apulia', 'North-Apulia', 'North-Apulia'], dtype=object)

In [7]:
X_train_oil, X_test_oil, y_train_oil, y_test_oil = train_test_split(X_oil, y_oil, test_size=0.3, random_state=42, stratify=y_oil)


## Topic 1 - Combined Grid Search [No tuning parameters]

#### Step 1 - Make a pipeline with a "hole" for the classifier.

In [8]:
(generic_cls :=
 Pipeline(steps = [
     ('scaler', StandardScaler()),
     ('classifier', None)  # <-- classifier goes here
 ])

)

,steps,"[('scaler', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True


#### Step 2 - Make a grid of classifiers (w/ no tuning parameters)

In [9]:
generic_cls.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()), ('classifier', None)],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'classifier': None,
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True}

In [10]:
(log_reg_grid :=
 {'classifier': [LogisticRegression(max_iter=10000),
                 OneVsRestClassifier(LogisticRegression(max_iter=10000)),
                 OneVsOneClassifier(LogisticRegression(max_iter=10000)),
               ],
 }
)

{'classifier': [LogisticRegression(max_iter=10000),
  OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
  OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000))]}

#### Step 3 - Set up and perform a combined grid search

In [31]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=4788)

In [51]:
(grid_search_log_reg :=
 GridSearchCV(generic_cls, log_reg_grid, cv=folds, scoring='balanced_accuracy', verbose=1, n_jobs=-1)
)

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [52]:
grid_search_log_reg.fit(X_train_oil, y_train_oil)

Fitting 5 folds for each of 3 candidates, totalling 15 fits


,estimator,"Pipeline(step...fier', None)])"
,param_grid,"{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


#### Evaluate the winning model on the test data.

In [53]:
# Bare bones --> use `.score` for accuracy.
grid_search_log_reg.score(X_test_oil, y_test_oil)

0.9037150213620802

In [54]:
# More complete --> Use CV on the test set to compute various metrics
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro',
           ]

(cv_test_scores :=
    cross_validate(grid_search_log_reg, X_test_oil, y_test_oil,
               cv=folds,
               scoring=metrics,
               verbose=1,
               n_jobs=-1,
               )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    1.5s finished


{'test_accuracy': np.float64(0.9010084033613446),
 'test_balanced_accuracy': np.float64(0.837037037037037),
 'test_f1_micro': np.float64(0.9010084033613446)}

## Topic 2 - Combined grid search with tuning parameters

#### Step 2 - Build a grid for each classifier + tuning parameters.

In [55]:
generic_cls.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()), ('classifier', None)],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'classifier': None,
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True}

In [56]:
KNeighborsClassifier().get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [57]:
(knn_alone := {'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': list(range(3, 12, 2)),
  })

{'classifier': [KNeighborsClassifier()],
 'classifier__n_neighbors': [3, 5, 7, 9, 11]}

In [58]:
OneVsRestClassifier(KNeighborsClassifier()).get_params()

{'estimator__algorithm': 'auto',
 'estimator__leaf_size': 30,
 'estimator__metric': 'minkowski',
 'estimator__metric_params': None,
 'estimator__n_jobs': None,
 'estimator__n_neighbors': 5,
 'estimator__p': 2,
 'estimator__weights': 'uniform',
 'estimator': KNeighborsClassifier(),
 'n_jobs': None,
 'verbose': 0}

In [59]:
(wrapped_knn :=
  {'classifier': [OneVsRestClassifier(KNeighborsClassifier()),
                 OneVsOneClassifier(KNeighborsClassifier()),
                ],
   'classifier__estimator__n_neighbors': list(range(3, 12, 2)),
  }
)

{'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
  OneVsOneClassifier(estimator=KNeighborsClassifier())],
 'classifier__estimator__n_neighbors': [3, 5, 7, 9, 11]}

In [60]:
(knn_grid :=
 [knn_alone, wrapped_knn]
)

[{'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 5, 7, 9, 11]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [3, 5, 7, 9, 11]}]

#### Step 3 - Set up and perform a grid search

In [61]:
(knn_grid_search :=
 GridSearchCV(generic_cls, knn_grid, cv=folds, scoring='accuracy', verbose=1)
)

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 5, ...]}, {'classifier': [OneVsRestClas...sClassifier()), OneVsOneClass...sClassifier())], 'classifier__estimator__n_neighbors': [3, 5, ...]}]"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [62]:
knn_grid_search.fit(X_train_oil, y_train_oil)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 5, ...]}, {'classifier': [OneVsRestClas...sClassifier()), OneVsOneClass...sClassifier())], 'classifier__estimator__n_neighbors': [3, 5, ...]}]"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


#### Step 4 - Evaluate performance on the test set

In [63]:
# Barebones --> Compute the accuracy score
knn_grid_search.score(X_test_oil, y_test_oil)

0.9593023255813954

In [64]:
# More complete --> Use CV on the test set to compute various metrics
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro',
           ]

(cv_test_scores :=
    cross_validate(knn_grid_search, X_test_oil, y_test_oil,
               cv=folds,
               scoring=metrics,
               verbose=1,
               n_jobs=-1,
               )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.9s finished


{'test_accuracy': np.float64(0.8836974789915966),
 'test_balanced_accuracy': np.float64(0.7925925925925925),
 'test_f1_micro': np.float64(0.8836974789915966)}

## Topic 3 - Combining multiple classifiers in a grid.

In [65]:
(combined_grid :=
 [log_reg_grid]  # Was just a dict
 + knn_grid      # Already a list of dict
)

[{'classifier': [LogisticRegression(max_iter=10000),
   OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
   OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000))]},
 {'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 5, 7, 9, 11]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [3, 5, 7, 9, 11]}]

In [66]:
(combined_grid_search :=
 GridSearchCV(generic_cls, combined_grid, cv=folds, scoring='accuracy', verbose=1)
)

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}, {'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 5, ...]}, ...]"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [67]:
combined_grid_search.fit(X_train_oil, y_train_oil)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}, {'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 5, ...]}, ...]"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


## <font color="red"> Exercise 1 </font>

1. Do a combined grid search on all 15 classifiers (5 classic classifiers + 5*OvR + 5*OvO).  
2. For `kNN`, add a number of `metric`s and `weights` to the grid.
3. Evaluate the performance of the winning model using CV and write a summary interpreting each metric.

In [68]:
KNeighborsClassifier().get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [72]:
log_reg_grid

{'classifier': [LogisticRegression(max_iter=10000),
  OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
  OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000))]}

In [74]:
(bayes_grid := 
    {'classifier': [GaussianNB(),
                 OneVsRestClassifier(GaussianNB()),
                 OneVsOneClassifier(GaussianNB()),
               ],
 })

{'classifier': [GaussianNB(),
  OneVsRestClassifier(estimator=GaussianNB()),
  OneVsOneClassifier(estimator=GaussianNB())]}

In [75]:
(QDA_grid := 
    {'classifier': [QuadraticDiscriminantAnalysis(),
                 OneVsRestClassifier(QuadraticDiscriminantAnalysis()),
                 OneVsOneClassifier(QuadraticDiscriminantAnalysis()),
               ],
 }
)

{'classifier': [QuadraticDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]}

In [76]:
(LDA_grid :=
    {'classifier': [LinearDiscriminantAnalysis(),
                 OneVsRestClassifier(LinearDiscriminantAnalysis()),
                 OneVsOneClassifier(LinearDiscriminantAnalysis()),
               ],
 }
)

{'classifier': [LinearDiscriminantAnalysis(),
  OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
  OneVsOneClassifier(estimator=LinearDiscriminantAnalysis())]}

In [ ]:
std_weights = ['uniform', 'distance']
std_metrics = ['minkowski', 'euclidean', 'manhattan']

In [ ]:
(knn_alone := {
    'classifier': [KNeighborsClassifier()],
    'classifier__n_neighbors': list(range(3, 12, 2)),
    'classifier__weights': std_weights,
    'classifier__metric': std_metrics,
    'classifier__p': [1, 2],  
})

{'classifier': [KNeighborsClassifier()],
 'classifier__n_neighbors': [3, 5, 7, 9, 11],
 'classifier__weights': ['uniform', 'distance'],
 'classifier__metric': ['minkowski', 'euclidean', 'manhattan'],
 'classifier__p': [1, 2]}

In [80]:
(wrapped_knn := {
    'classifier': [
        OneVsRestClassifier(KNeighborsClassifier()),
        OneVsOneClassifier(KNeighborsClassifier()),
    ],
    'classifier__estimator__n_neighbors': list(range(3, 12, 2)),
    'classifier__estimator__weights': std_weights,
    'classifier__estimator__metric': std_metrics,
    'classifier__estimator__p': [1, 2],
})


{'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
  OneVsOneClassifier(estimator=KNeighborsClassifier())],
 'classifier__estimator__n_neighbors': [3, 5, 7, 9, 11],
 'classifier__estimator__weights': ['uniform', 'distance'],
 'classifier__estimator__metric': ['minkowski', 'euclidean', 'manhattan'],
 'classifier__estimator__p': [1, 2]}

In [83]:
(knn_full_grid := [knn_alone, wrapped_knn])

[{'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 5, 7, 9, 11],
  'classifier__weights': ['uniform', 'distance'],
  'classifier__metric': ['minkowski', 'euclidean', 'manhattan'],
  'classifier__p': [1, 2]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneClassifier(estimator=KNeighborsClassifier())],
  'classifier__estimator__n_neighbors': [3, 5, 7, 9, 11],
  'classifier__estimator__weights': ['uniform', 'distance'],
  'classifier__estimator__metric': ['minkowski', 'euclidean', 'manhattan'],
  'classifier__estimator__p': [1, 2]}]

In [85]:
(combined_full_grids := 
 [log_reg_grid, bayes_grid, QDA_grid, LDA_grid]
 + knn_full_grid
)

[{'classifier': [LogisticRegression(max_iter=10000),
   OneVsRestClassifier(estimator=LogisticRegression(max_iter=10000)),
   OneVsOneClassifier(estimator=LogisticRegression(max_iter=10000))]},
 {'classifier': [GaussianNB(),
   OneVsRestClassifier(estimator=GaussianNB()),
   OneVsOneClassifier(estimator=GaussianNB())]},
 {'classifier': [QuadraticDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=QuadraticDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=QuadraticDiscriminantAnalysis())]},
 {'classifier': [LinearDiscriminantAnalysis(),
   OneVsRestClassifier(estimator=LinearDiscriminantAnalysis()),
   OneVsOneClassifier(estimator=LinearDiscriminantAnalysis())]},
 {'classifier': [KNeighborsClassifier()],
  'classifier__n_neighbors': [3, 5, 7, 9, 11],
  'classifier__weights': ['uniform', 'distance'],
  'classifier__metric': ['minkowski', 'euclidean', 'manhattan'],
  'classifier__p': [1, 2]},
 {'classifier': [OneVsRestClassifier(estimator=KNeighborsClassifier()),
   OneVsOneCl

In [87]:
(combined_full_grid_search := GridSearchCV(
    generic_cls,
    combined_full_grids,
    cv=folds,
    scoring='balanced_accuracy',
    verbose=1
))

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}, {'classifier': [GaussianNB(), OneVsRestClas...=GaussianNB()), ...]}, ...]"
,scoring,'balanced_accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [88]:
combined_full_grid_search.fit(X_train_oil, y_train_oil)

Fitting 5 folds for each of 192 candidates, totalling 960 fits


c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
c:\Users\kh6102sj\AppData\Local\anaconda3\envs\polars\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_p

,estimator,"Pipeline(step...fier', None)])"
,param_grid,"[{'classifier': [LogisticRegre...ax_iter=10000), OneVsRestClas...x_iter=10000)), ...]}, {'classifier': [GaussianNB(), OneVsRestClas...=GaussianNB()), ...]}, ...]"
,scoring,'balanced_accuracy'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [89]:
combined_full_grid_search.score(X_test_oil, y_test_oil)

0.932057267351385

In [90]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro',
           ]

(cv_test_scores :=
    cross_validate(combined_full_grid_search, X_test_oil, y_test_oil,
               cv=folds,
               scoring=metrics,
               verbose=1,
               n_jobs=-1,
               )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   52.1s finished


{'test_accuracy': np.float64(0.9010084033613446),
 'test_balanced_accuracy': np.float64(0.8444444444444443),
 'test_f1_micro': np.float64(0.9010084033613446)}

In [94]:
best_model = combined_full_grid_search.best_estimator_

metrics = ['accuracy', 'balanced_accuracy', 'f1_micro']
(cv_test_scores := (
    cross_validate(
        best_model, X_train_oil, y_train_oil,
        cv=folds, scoring=metrics, n_jobs=-1
    )
    >> rec.subset([f'test_{m}' for m in metrics])
    >> rec.map(np.mean)
))

{'test_accuracy': np.float64(0.9375),
 'test_balanced_accuracy': np.float64(0.9046579091406677),
 'test_f1_micro': np.float64(0.9375)}

In [ ]:
best_model.score(X_test_oil, y_test_oil)

0.9534883720930233

In [97]:
# Exercise 1 summary helper
print('Best CV score (balanced_accuracy):', combined_full_grid_search.best_score_)
print('Best params:', combined_full_grid_search.best_params_)
print('Best estimator:', combined_full_grid_search.best_estimator_)
print('Train CV metrics (mean):', cv_test_scores)
print('Holdout test accuracy:', best_model.score(X_test_oil, y_test_oil))

Best CV score (balanced_accuracy): 0.9046579091406677
Best params: {'classifier': QuadraticDiscriminantAnalysis()}
Best estimator: Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', QuadraticDiscriminantAnalysis())])
Train CV metrics (mean): {'test_accuracy': np.float64(0.9375), 'test_balanced_accuracy': np.float64(0.9046579091406677), 'test_f1_micro': np.float64(0.9375)}
Holdout test accuracy: 0.9534883720930233



<font color = "orange">
The combined grid search selected QDA as the best model, with a cross-validated balanced accuracy of 0.9047. Its mean CV accuracy and F1-micro were both 0.9375, and on the holdout test set it reached 0.9535 accuracy. These scores suggest strong overall performance with good class balance, so QDA was the strongest choice among the tested classic, wrapped, and KNN-based models.
</font>